# Lesson 21: Capstone projects and your AI engineering career

*Worked solutions for [the lesson page](../../lessons/21-capstones-career.html). The exercise version without solutions is `../21-capstones-career.ipynb`.*

Prove your skills by building one end-to-end system from five capstone specifications with rubrics, then turn the work into a portfolio, prepare for interviews with worked questions, and plan your next ninety days.

**Runs on:** Any computer. This notebook is a capstone workbook.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy

## The five capstones

### Hands-on: scope your capstone

Write a one-page project brief before writing any code:

1. One sentence: who it is for and what it does.
2. Three measurable success criteria.
3. Your data sources and how you will build the eval set.
4. The simplest architecture that could work, as a sketch.
5. The three riskiest assumptions, and how you will test each in the first week.
6. What you will deliberately **not** build.

#### Solution: an example brief (Capstone A)

**Who and what**: answers calibration and maintenance questions for one instrument model from its 420-page service manual and 60 service bulletins, with page citations, for field technicians.

**Success criteria**: (1) correct page in the top 6 retrieved for at least 90% of 60 answerable test questions; (2) faithful, correctly cited answers for at least 85% as judged by a calibrated judge; (3) says "not in the documents" for at least 9 of 10 unanswerable questions; median time to first token under 2 seconds.

**Data and eval**: the PDF manual and bulletins; 40 questions from two experienced technicians plus 20 reviewed synthetic questions and 10 unanswerable ones; gold page numbers recorded for each.

**Architecture**: Docling parsing, section-level chunks with title and section path, BM25 plus embeddings with RRF, cross-encoder reranking to 6, a mid-tier model with the grounded prompt, FastAPI streaming, JSON-line traces.

**Riskiest assumptions**: tables of torque values parse correctly (test 10 tables on day one); technicians phrase questions very differently from the manual (collect real questions early); bulletins that supersede manual sections can be detected (check metadata).

**Not building**: user accounts, a mobile app, multi-manual support, fine-tuning.

## Interview preparation

**Practice question:** Explain the bias-variance trade-off and how you diagnose it.

Bias is error from overly simple assumptions (underfitting: high training and validation error); variance is error from sensitivity to the particular training data (overfitting: low training error, high validation error). Diagnose by comparing training and validation error; fix underfitting with more capacity or better features, overfitting with more data, regularisation, simpler models or early stopping (lesson 03).

**Practice question:** Walk through scaled dot-product attention. Why divide by √d_k?

Each token's query is dotted with every key, scaled, softmaxed into weights, and used to average the values: softmax(QKᵀ/√d_k)V. Dot products of random d-dimensional vectors have standard deviation √d, so without scaling large dimensions push softmax into saturation and gradients vanish (lesson 08).

**Practice question:** What is the KV cache, and why does it matter for serving?

During generation, keys and values of previous tokens are stored so each step only computes the new token's projections. It makes generation efficient but consumes memory proportional to layers × KV heads × head dimension × context length × concurrent sequences, which often limits how many users a GPU can serve (lessons 08 and 17).

**Practice question:** How would you reduce hallucinations in a customer-facing assistant?

Ground answers in retrieved sources with citations; instruct and allow abstention; use structured outputs and tools for facts and arithmetic; lower temperature for factual tasks; add a faithfulness check step for high-risk answers; evaluate faithfulness and abstention on a golden set and monitor them in production (lessons 13 and 15).

**Practice question:** RAG versus fine-tuning versus long context: how do you choose?

RAG for knowledge that is large, changing, access-controlled or needs citations. Fine-tuning for behaviour, format, style, or making a small model do a narrow task cheaply. Long context with caching for small, stable collections where simplicity matters. Often combined (lessons 13 and 16).

**Practice question:** Explain LoRA. Why does it work, and what does it save?

Freeze the pretrained weights and learn a low-rank update BA (rank r much smaller than the layer dimensions), scaled by α/r, with B initialised to zero. Weight updates during fine-tuning have low intrinsic rank, so this captures most of the benefit while training under 1% of the parameters, cutting optimiser memory dramatically and producing small swappable adapters; QLoRA also stores the base model in 4-bit (lesson 16).

**Practice question:** How do you evaluate an LLM feature when outputs are open-ended?

A golden dataset from real usage with hard and negative cases; code graders for anything rule-based; LLM judges on specific criteria with clear rubrics, structured verdicts, order swapping for pairwise comparisons, and calibration against human grades; per-slice reporting with confidence intervals and fixed/broken case analysis; online signals in production (lesson 15).

**Practice question:** What is prompt injection and how do you defend against it?

Instructions embedded in content the model reads (direct from users, or indirect via documents, web pages, tool results) that try to override the system's instructions. No complete fix exists, so defend in depth: separate and label untrusted content, but above all limit impact with least-privilege tools, human approval for consequential actions, output filtering, safe rendering and avoiding agents that combine untrusted input, private data and external communication (lesson 18).

**Practice question:** Explain RLHF and DPO.

RLHF: collect human preference comparisons, train a reward model to predict them, then optimise the language model with reinforcement learning to maximise reward with a KL penalty that keeps it close to the supervised model. DPO: skip the reward model and RL, and optimise the preference pairs directly with a classification-style loss relative to a frozen reference model; simpler and more stable (lessons 09 and 16).

**Practice question:** Your RAG system's answers got worse after a change. How do you debug it?

Reproduce on the eval set and list newly broken cases. For each, check the trace: was the right chunk retrieved (parsing, chunking, embedding, filters)? Was it ranked high enough (reranker)? Was it in the prompt but ignored or contradicted (prompt, model, context length)? Fix the stage responsible, re-run the full eval, and add the cases to the regression set (lessons 13, 15 and 17).

## Career paths and the next ninety days

### Hands-on: your ninety-day plan

Write a plan with three 30-day blocks. For each block: one thing to build, one topic to go deeper on, one thing to publish, and one way to get feedback from people more experienced than you. Put it at the top of your learning log, and schedule a review at day 30.

#### Solution: an example plan

**Days 1 to 30**: build Capstone A on the service manuals you use at work (with permission, or on public equivalents); go deeper on retrieval evaluation; publish the repository with a results table; get feedback by demoing to two colleagues who would actually use it.

**Days 31 to 60**: extend it with an MCP server and an agent mode (Capstone B ideas); go deeper on security and prompt injection; publish a write-up of what worked and what did not; get feedback by asking a security-minded colleague to attack it.

**Days 61 to 90**: distil the most common query type into a fine-tuned small model running offline (Capstone D ideas); go deeper on quantisation and serving; publish a cost-and-quality comparison; get feedback through an open-source contribution or a talk at a local meetup.

## Capstone workbook

Keep this notebook (or a copy of it) at the root of your capstone repository.

### Evaluation set first

Write your cases before your system. Start with 30; grow to 100 or more.

In [ ]:
import json

eval_cases = [
    {"id": "q-001", "input": "Replace with a real question", "expected": "what a correct answer must contain",
     "tags": ["lookup"]},
    {"id": "q-002", "input": "A question your data cannot answer", "expected": "NOT_IN_DOCS", "tags": ["unanswerable"]},
]
with open("eval_cases.jsonl", "w") as f:
    for c in eval_cases:
        f.write(json.dumps(c) + "\n")
print(len(eval_cases), "cases written")

### Decision log

| Date | Decision | Alternatives considered | Evidence | Result |
|---|---|---|---|---|
| | | | | |

### Results table for the write-up

| System | Metric 1 | Metric 2 | Cost per 1,000 requests | p50 latency |
|---|---|---|---|---|
| Baseline | | | | |
| Final | | | | |

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/21-capstones-career.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.